# 附：Codes .{unnumbered}

本 Notebook 用于生成 `05_truncated_twopart_lec_v3.ipynb` 中调用的模拟数据和图片。运行后会生成：

- `./data/limitdep_credit_alt_sim.csv`
- `./data/two_part_ame.csv`
- `./data/two_part_predictions.csv`
- `./figs/limit_dep_alt_fig01_model_map.png`
- `./figs/limit_dep_alt_fig02_two_part_mechanism.png`
- `./figs/limit_dep_alt_fig03_tobit_vs_twopart.png`
- `./figs/limit_dep_alt_fig04_hurdle_mechanism.png`
- `./figs/limit_dep_alt_fig05_zero_inflated.png`
- `./figs/limit_dep_alt_fig06_frm_vs_ols.png`

In [1]:
# ============================================================
# 0. 全局设置
# ============================================================

import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.font_manager as fm
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import stats
from scipy.special import expit
import statsmodels.api as sm

FIG_DIR = Path("./figs")
DATA_DIR = Path("./data")
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# 中文字体设置，避免图片乱码
font_paths = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
    "C:/Windows/Fonts/simhei.ttf",
    "C:/Windows/Fonts/msyh.ttc",
]

for fp in font_paths:
    if os.path.exists(fp):
        fm.fontManager.addfont(fp)
        font_prop = fm.FontProperties(fname=fp)
        plt.rcParams["font.family"] = font_prop.get_name()
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

def savefig(fig, name):
    """
    同时保存 png 和 svg，方便讲义、网页和推文使用。
    """
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)

In [2]:
# ============================================================
# 1. 生成 Two-part / Hurdle / FRM 的企业信贷数据
# ============================================================

def simulate_credit_alt(n=3500, seed=20260430):
    """
    生成本章使用的企业信贷模拟数据。

    主线变量：
    - loan_amt: 企业银行贷款金额；
    - D: 是否获得贷款；
    - loan_count_hurdle: 适合 Hurdle model 的贷款笔数；
    - loan_count_zi: 适合 Zero-inflated model 的贷款笔数；
    - loan_share: 适合 Fractional response model 的贷款比例。
    """
    r = np.random.default_rng(seed)

    # 投资机会：主要影响获得贷款后的贷款金额
    opportunity = r.normal(0, 1, n)

    # 抵押能力：既影响能否获得贷款，也影响获贷后的额度
    collateral = (r.beta(2.2, 2.0, n) - 0.52) / 0.22

    # 银行可得性：主要影响能否进入信贷关系
    bank_access = r.normal(0, 1, n)

    # 风险与规模：用于扩展分析
    risk = r.normal(0, 1, n)
    size = r.normal(0, 1, n)

    # 第一部分：是否获得贷款
    eta_d = -0.35 + 1.10 * collateral + 0.95 * bank_access - 0.35 * risk
    p_loan = expit(eta_d)
    D = r.binomial(1, p_loan, n)

    # 第二部分：获得贷款后的贷款金额
    eps = r.normal(0, 0.55, n)
    log_loan_pos = 2.20 + 0.42 * collateral + 0.38 * opportunity + 0.18 * size + eps
    loan_pos_amt = np.exp(log_loan_pos)
    loan_amt = D * loan_pos_amt

    # Hurdle count：先跨过门槛，再生成正计数
    eta_h = -0.60 + 1.0 * collateral + 0.7 * bank_access + 0.4 * opportunity
    p_h = expit(eta_h)
    H = r.binomial(1, p_h, n)
    lam_h = np.exp(0.65 + 0.35 * collateral + 0.30 * opportunity)
    count_h = np.zeros(n, dtype=int)

    for i in range(n):
        if H[i] == 1:
            val = 0
            while val == 0:
                val = r.poisson(lam_h[i])
            count_h[i] = val

    # Zero-inflated count：0 有两个来源
    pi = expit(0.15 - 0.95 * bank_access - 0.75 * collateral)
    structural_zero = r.binomial(1, pi, n)
    lam_zi = np.exp(0.45 + 0.45 * opportunity + 0.35 * collateral)
    count_zi = np.where(structural_zero == 1, 0, r.poisson(lam_zi))

    # Fractional response：银行贷款占总负债比例
    mu_share = expit(-0.20 + 0.75 * collateral + 0.45 * bank_access - 0.55 * risk + 0.25 * opportunity)
    precision = 18
    a = np.clip(mu_share * precision, 0.5, None)
    b = np.clip((1 - mu_share) * precision, 0.5, None)
    loan_share = r.beta(a, b)

    return pd.DataFrame({
        "firm_id": np.arange(1, n + 1),
        "opportunity": opportunity,
        "collateral": collateral,
        "bank_access": bank_access,
        "risk": risk,
        "size": size,
        "p_loan_true": p_loan,
        "D": D,
        "loan_amt": loan_amt,
        "log_loan_pos": np.where(D == 1, np.log(np.maximum(loan_amt, 1e-9)), np.nan),
        "loan_count_hurdle": count_h,
        "loan_count_zi": count_zi,
        "structural_zero": structural_zero,
        "loan_share": loan_share,
    })

df = simulate_credit_alt()
df.to_csv(DATA_DIR / "limitdep_credit_alt_sim.csv", index=False, encoding="utf-8-sig")

df.head()

,firm_id,opportunity,collateral,bank_access,risk,size,p_loan_true,D,loan_amt,log_loan_pos,loan_count_hurdle,loan_count_zi,structural_zero,loan_share
0,1,-0.621382,-1.497976,-1.236858,0.292090,-0.378711,0.036438,0,0.0,NaN,0,0,1,0.040189
1,2,0.861608,1.416697,-0.557567,0.293363,0.849885,0.640147,0,0.0,NaN,5,4,0,0.445002
2,3,-0.423961,-0.975556,-0.468398,-1.617673,0.234908,0.213845,0,0.0,NaN,1,0,1,0.293465
3,4,1.151212,-2.161833,0.298261,-0.159171,-2.082310,0.084019,0,0.0,NaN,0,0,1,0.135474
4,5,-0.084264,-0.147204,-0.140035,-0.704008,1.083729,0.401660,0,0.0,NaN,0,0,1,0.576498


In [3]:
# ============================================================
# 2. Two-part model 的估计、预测与平均边际效应
# ============================================================

def fit_two_part(df, z_vars, x_vars, y_col="loan_amt"):
    """
    估计 Two-part model。

    第一部分：Logit，解释是否获得贷款；
    第二部分：log-OLS，解释获得贷款后的贷款金额；
    同时计算 smearing correction，用于把 log 预测转回 level。
    """
    D = (df[y_col] > 0).astype(int)

    # 第一部分：是否获得贷款
    Z = sm.add_constant(df[z_vars])
    part1 = sm.Logit(D, Z).fit(disp=False)

    # 第二部分：正贷款金额
    pos = D == 1
    X_pos = sm.add_constant(df.loc[pos, x_vars])
    log_y = np.log(df.loc[pos, y_col])
    part2 = sm.OLS(log_y, X_pos).fit()

    # Duan smearing factor
    smear = np.exp(part2.resid).mean()

    return {
        "part1": part1,
        "part2": part2,
        "smear": smear,
        "z_vars": z_vars,
        "x_vars": x_vars,
    }

def predict_two_part(models, df):
    """
    计算 Three objects：
    1. p_hat: 获得贷款的预测概率；
    2. mu_hat: 获得贷款后的预测条件均值；
    3. ey_hat: 非条件期望 p_hat * mu_hat。
    """
    Z = sm.add_constant(df[models["z_vars"]])
    X = sm.add_constant(df[models["x_vars"]])

    p = models["part1"].predict(Z)
    mu = np.exp(models["part2"].predict(X)) * models["smear"]

    return pd.DataFrame({
        "p_hat": p,
        "mu_hat": mu,
        "ey_hat": p * mu,
    })

def ame_two_part(models, df, varname):
    """
    计算 Two-part model 的平均边际效应。

    如果变量同时进入两个方程，边际效应可以分解为：
    - probability_channel: 概率渠道；
    - intensity_channel: 强度渠道。
    """
    pred = predict_two_part(models, df)
    p = pred["p_hat"].to_numpy()
    mu = pred["mu_hat"].to_numpy()

    gamma = models["part1"].params.get(varname, 0.0)
    beta = models["part2"].params.get(varname, 0.0)

    # Logit 第一部分的密度项为 p(1-p)
    probability_channel = p * (1 - p) * gamma * mu
    intensity_channel = p * mu * beta

    return {
        "varname": varname,
        "probability_channel": probability_channel.mean(),
        "intensity_channel": intensity_channel.mean(),
        "total_ame": (probability_channel + intensity_channel).mean(),
    }

models = fit_two_part(
    df,
    z_vars=["collateral", "bank_access"],
    x_vars=["collateral", "opportunity"],
    y_col="loan_amt"
)

pred = predict_two_part(models, df)
ame = pd.DataFrame([
    ame_two_part(models, df, "collateral"),
    ame_two_part(models, df, "bank_access"),
    ame_two_part(models, df, "opportunity"),
])

pd.concat([df[["firm_id", "loan_amt", "D"]], pred], axis=1).to_csv(
    DATA_DIR / "two_part_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

ame.to_csv(DATA_DIR / "two_part_ame.csv", index=False, encoding="utf-8-sig")

ame.round(4)

,varname,probability_channel,intensity_channel,total_ame
0,collateral,2.5864,2.8904,5.4767
1,bank_access,2.0638,0.0000,2.0638
2,opportunity,0.0000,2.6644,2.6644


In [4]:
# ============================================================
# 3. 图 1：模型地图
# ============================================================

def draw_box(ax, xy, text, fc, ec, w, h=0.82, dashed=False, fontsize=12):
    x, y = xy
    box = FancyBboxPatch(
        (x - w / 2, y - h / 2),
        w,
        h,
        boxstyle="round,pad=0.06,rounding_size=0.08",
        fc=fc,
        ec=ec,
        lw=1.25,
        linestyle="--" if dashed else "-"
    )
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)
    return {"x": x, "y": y, "w": w, "h": h}

def edge(box, side):
    if side == "right":
        return (box["x"] + box["w"] / 2, box["y"])
    if side == "left":
        return (box["x"] - box["w"] / 2, box["y"])
    if side == "top":
        return (box["x"], box["y"] + box["h"] / 2)
    if side == "bottom":
        return (box["x"], box["y"] - box["h"] / 2)

def arrow(ax, b1, side1, b2, side2, rad=0.0):
    arr = FancyArrowPatch(
        edge(b1, side1),
        edge(b2, side2),
        arrowstyle="-|>",
        mutation_scale=12,
        lw=1.15,
        color="0.25",
        shrinkA=8,
        shrinkB=8,
        connectionstyle=f"arc3,rad={rad}"
    )
    ax.add_patch(arr)

colors = {
    "blue": ("#D7EAFB", "#5B8FC6"),
    "yellow": ("#FFF1C9", "#D5A537"),
    "green": ("#DFF0D8", "#6BA35D"),
    "red": ("#FADBD8", "#B7645C"),
    "purple": ("#E8DAEF", "#8E6AA7"),
    "orange": ("#FDEBD0", "#D49A45"),
    "teal": ("#D5F5E3", "#4E9A73"),
    "gray": ("#F2F3F4", "#8A8A8A"),
}

fig, ax = plt.subplots(figsize=(12.6, 6.0))
ax.set_xlim(0, 12.6)
ax.set_ylim(0, 6.0)
ax.axis("off")

b0 = draw_box(ax, (1.25, 5.30), "因变量受限\n或有大量 0", *colors["blue"], w=1.85)
b1 = draw_box(ax, (3.85, 5.30), "样本是否\n仍在数据中？", *colors["yellow"], w=1.95)
b2 = draw_box(ax, (6.85, 5.30), "否：样本被截断\nTruncated regression", *colors["red"], w=3.00)
b3 = draw_box(ax, (3.85, 4.05), "是：0 或 missing\n如何产生？", *colors["green"], w=2.35)
b4 = draw_box(ax, (1.35, 2.75), "边界外取值被归并\nTobit", *colors["purple"], w=2.20)
b5 = draw_box(ax, (4.05, 2.75), "0 是真实选择\nTwo-part model", *colors["blue"], w=2.35)
b6 = draw_box(ax, (7.05, 2.75), "结果是计数\nHurdle / Zero-inflated", *colors["orange"], w=2.65)
b7 = draw_box(ax, (10.05, 2.75), "结果是比例\nFractional response", *colors["teal"], w=2.45)
b8 = draw_box(
    ax,
    (6.50, 1.10),
    "结果变量只在被选择样本中可观测\nHeckman selection (Chap. 06)",
    *colors["gray"],
    w=4.45,
    h=0.88,
    dashed=True
)

arrow(ax, b0, "right", b1, "left")
arrow(ax, b1, "right", b2, "left")
arrow(ax, b1, "bottom", b3, "top")
arrow(ax, b3, "left", b4, "top", rad=0.05)
arrow(ax, b3, "bottom", b5, "top")
arrow(ax, b5, "right", b6, "left")
arrow(ax, b6, "right", b7, "left")

ax.text(
    0.25,
    0.25,
    "注：灰色虚线框为下一章样本选择模型的入口，本章不展开。",
    fontsize=10,
    color="0.35"
)

savefig(fig, "limit_dep_alt_fig01_model_map")

In [5]:
# ============================================================
# 4. 图 2：Two-part 机制图
# ============================================================

fig, ax = plt.subplots(figsize=(8.4, 4.7))
ax.axis("off")
ax.set_xlim(0, 8.4)
ax.set_ylim(0, 4.7)

b1 = draw_box(ax, (1.4, 3.2), "抵押能力\ncollateral", "#E8F6F3", "#45B39D", w=1.6, fontsize=12)
b2 = draw_box(ax, (1.4, 1.7), "银行可得性\nbank_access", "#FEF9E7", "#D4AC0D", w=1.8, fontsize=12)
b3 = draw_box(ax, (1.4, 0.45), "投资机会\nopportunity", "#FDEDEC", "#CD6155", w=1.7, fontsize=12)

c1 = draw_box(ax, (4.2, 2.55), "第一部分\n是否获得贷款\nPr(D=1|z)", "#D6EAF8", "#5DADE2", w=2.1, h=1.05, fontsize=12)
c2 = draw_box(ax, (4.2, 0.95), "第二部分\n获贷后贷多少\nE(B|B>0,x)", "#EBDEF0", "#AF7AC5", w=2.3, h=1.05, fontsize=12)

out = draw_box(ax, (7.0, 1.75), "非条件期望\nE(B|z,x)=p·μ+", "#D5F5E3", "#58D68D", w=2.0, fontsize=12)

arrow(ax, b1, "right", c1, "left")
arrow(ax, b1, "right", c2, "left")
arrow(ax, b2, "right", c1, "left")
arrow(ax, b3, "right", c2, "left")
arrow(ax, c1, "right", out, "left")
arrow(ax, c2, "right", out, "left")

ax.set_title("Two-part model：概率渠道与强度渠道", fontsize=14, pad=6)

savefig(fig, "limit_dep_alt_fig02_two_part_mechanism")

In [6]:
# ============================================================
# 5. 图 3：Tobit 与 Two-part 机制对比
# ============================================================

opp_grid = np.linspace(-2.5, 2.5, 160)

# Tobit：同一个潜在机制同时影响概率与正值大小
xb_tobit = -0.2 + 0.9 * opp_grid
p_tobit = stats.norm.cdf(xb_tobit / 0.9)
ey_tobit = p_tobit * xb_tobit + 0.9 * stats.norm.pdf(xb_tobit / 0.9)

# Two-part：这里为了说明机制，让 opportunity 主要影响强度
p_tpm = expit(-0.2 + 0.8 * np.zeros_like(opp_grid))
mu_tpm = np.exp(1.0 + 0.38 * opp_grid)
ey_tpm = p_tpm * mu_tpm / np.mean(mu_tpm) * np.max(ey_tobit)

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(opp_grid, ey_tobit, lw=2.2, label="Tobit：同一潜在机制")
ax.plot(opp_grid, ey_tpm, lw=2.2, ls="--", label="Two-part：概率与强度可分开")

ax.set_xlabel("投资机会 opportunity")
ax.set_ylabel("预测贷款金额")
ax.set_title("同一个贷款背景，可以对应不同数据生成机制")
ax.legend(frameon=False)

savefig(fig, "limit_dep_alt_fig03_tobit_vs_twopart")

In [7]:
# ============================================================
# 6. 图 4 和图 5：Hurdle 与 Zero-inflated 的零值机制
# ============================================================

# Hurdle：0 与正计数分开
fig, ax = plt.subplots(figsize=(7.6, 4.3))
vals = df["loan_count_hurdle"]
counts = pd.Series(vals).value_counts().sort_index()

ax.bar(counts.index, counts.values / counts.values.sum(), width=0.75, alpha=0.75)
ax.axvline(0.5, color="0.35", ls="--", lw=1.2)
ax.text(
    0.62,
    ax.get_ylim()[1] * 0.80,
    "跨过 0 的门槛后\n正计数来自零截断分布",
    fontsize=11
)

ax.set_xlabel("贷款笔数")
ax.set_ylabel("样本占比")
ax.set_title("Hurdle model：先决定是否跨过 0，再决定正计数")

savefig(fig, "limit_dep_alt_fig04_hurdle_mechanism")

# Zero-inflated：0 有两个来源
fig, ax = plt.subplots(figsize=(7.6, 4.3))
vals = df["loan_count_zi"]
counts = pd.Series(vals).value_counts().sort_index().loc[:8]

ax.bar(counts.index, counts.values / len(vals), width=0.75, alpha=0.75)

zero_share = (vals == 0).mean()
struct_share = ((df["loan_count_zi"] == 0) & (df["structural_zero"] == 1)).mean()

ax.text(
    1.1,
    ax.get_ylim()[1] * 0.85,
    f"0 的总比例约为 {zero_share:.1%}\n其中一部分是结构性 0",
    fontsize=11
)

ax.set_xlabel("贷款笔数")
ax.set_ylabel("样本占比")
ax.set_title("Zero-inflated model：0 可以来自两个来源")

savefig(fig, "limit_dep_alt_fig05_zero_inflated")

In [8]:
# ============================================================
# 7. 图 6：FRM 与 OLS 的预测对比
# ============================================================

X = sm.add_constant(df[["collateral", "bank_access", "risk"]])

# OLS：可能给出 [0,1] 之外的预测值
ols_frm = sm.OLS(df["loan_share"], X).fit()

# FRM：这里用 GLM + Binomial family 作为 fractional logit 的估计方式
glm_frm = sm.GLM(df["loan_share"], X, family=sm.families.Binomial()).fit()

pred_ols = ols_frm.predict(X)
pred_glm = glm_frm.predict(X)

fig, ax = plt.subplots(figsize=(7.6, 4.3))

ax.scatter(df["collateral"], pred_ols, s=10, alpha=0.22, label="OLS 预测值")
ax.scatter(df["collateral"], pred_glm, s=10, alpha=0.22, label="FRM 预测值")

ax.axhline(0, color="0.3", ls="--", lw=1.0)
ax.axhline(1, color="0.3", ls="--", lw=1.0)

ax.set_xlabel("抵押能力 collateral")
ax.set_ylabel("预测贷款比例")
ax.set_title("Fractional response：预测值自然落在 [0,1]")
ax.legend(frameon=False)

savefig(fig, "limit_dep_alt_fig06_frm_vs_ols")

# 运行完成检查

如果上述代码顺利运行，当前目录下会生成 `./data/` 和 `./figs/` 中的所有配套文件。讲义正文只需要引用生成后的图片和数据结果。